# Polarimetric SAR and speckle — Long & Ulaby Chapter 5
The examples use the H,V channel order and $e_r^H S e_t$ synthesis convention.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from navasar.polarimetry import scattering_matrix, polarization_vector, synthesize_scattering, pauli_vector, covariance_matrix, entropy_anisotropy, simulate_complex_speckle
rng = np.random.default_rng(5)

## Polarization synthesis
Rotate a linear co-polarized antenna across a reciprocal target.

In [ ]:
S = scattering_matrix(1.0, .15j, .15j, .35*np.exp(.5j))
orientation = np.linspace(0, 180, 361)
power = np.array([abs(synthesize_scattering(S, polarization_vector(a)))**2 for a in orientation])
plt.plot(orientation, 10*np.log10(power)); plt.xlabel('Linear polarization orientation (degree)')
plt.ylabel('Synthesized power (dB)'); plt.grid(alpha=.3);

## Covariance, entropy, and correlated speckle
Complex-Gaussian samples should recover their generating covariance as the number of looks grows.

In [ ]:
target_C = np.array([[1, .25+.1j, 0], [.25-.1j, .5, .08j], [0, -.08j, .2]])
samples = simulate_complex_speckle(target_C, 50_000, rng)
measured_C = covariance_matrix(samples)
H, A = entropy_anisotropy(measured_C)
print('Measured covariance:\n', np.round(measured_C, 3))
print(f'Entropy={H:.3f}, anisotropy={A:.3f}')

## Radar equation, conventions and despeckling
Multilooking and local-statistics filtering reduce fluctuations in different ways.

In [ ]:
from navasar.polarimetry import radar_received_power, distributed_target_rcs, multilook_intensity, lee_filter, polarimetric_parameters
rcs = distributed_target_rcs(.1, 100, 35)
power_w = radar_received_power(1000, 100, .056, rcs, 800_000)
speckled = multilook_intensity(1, looks=1, size=(160,160), rng=rng)
filtered = lee_filter(speckled, window_size=7)
fig, ax = plt.subplots(1,2,figsize=(8,4)); ax[0].imshow(speckled,cmap='gray'); ax[0].set_title('Single look'); ax[1].imshow(filtered,cmap='gray'); ax[1].set_title('Lee filtered')
for a in ax: a.axis('off')
print('Received power (W):', power_w); print(polarimetric_parameters(measured_C))